# 02 — Extended EDA: customer behaviour and data quality

Complements `01_eda.py` (seasonality, category mix, style lifetime). Everything here is computed from the full
transactions (31.8M rows), articles (105,542) and customers files with DuckDB and pandas.
Run order: `scripts/convert_data.py` first; `customers.csv` is read from the raw-data folder.

In [1]:
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parents[1]))  # notebook lives in data_science/notebooks/
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

import config
from data_science import viz

viz.apply_style()
pd.set_option("display.width", 140, "display.max_columns", 20)
RAW = Path(os.getenv("HM_RAW_DIR") or config.DATA_DIR.parent)
con = duckdb.connect(config={"memory_limit": "3GB"})  # progress bar is off by default in Jupyter
TX = f"read_parquet('{config.TX_GLOB}')"
ART = f"read_parquet('{config.ARTICLES_PATH}')"
CUST = f"read_csv_auto('{(RAW / 'customers.csv').as_posix()}', types={{'customer_id': 'VARCHAR'}})"
# transactions store cust = CAST(hash(customer_id) AS UBIGINT) (scripts/convert_data.py); same hash here
con.execute(f"CREATE TEMP TABLE cust AS SELECT *, CAST(hash(customer_id) AS UBIGINT) AS cust FROM {CUST}")
con.execute(f"""CREATE TEMP TABLE c_tx AS SELECT cust, count(*) AS n_tx, count(DISTINCT t_dat) AS n_days,
               avg(CAST(channel = 2 AS DOUBLE)) AS online_share FROM {TX} GROUP BY cust""")
q = lambda sql: con.execute(sql).df()
q("SELECT (SELECT count(*) FROM cust) AS customers, (SELECT count(*) FROM c_tx) AS buying_customers, "
  "(SELECT count(*) FROM c_tx WHERE cust NOT IN (SELECT cust FROM cust)) AS buyers_not_in_customers_csv")

,customers,buying_customers,buyers_not_in_customers_csv
0,1371980,1362281,0


## 1. Customer behaviour
### Age: distribution and missing share

In [2]:
age = q("SELECT age FROM cust")["age"]
print(f"customers: {len(age):,} | age missing: {age.isna().sum():,} ({age.isna().mean():.2%})")
print(age.describe().round(1).to_string())
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.hist(age.dropna(), bins=range(16, 100), color=viz.SERIES[0])
ax.set_xlabel("age"); ax.set_ylabel("customers")
ax.set_title("Customer age (customers.csv)"); plt.show()
by_age = age.value_counts().sort_index()
print(f"main peak: age {by_age.idxmax()} ({by_age.max():,}); trough 35-45 at age {by_age.loc[35:45].idxmin()} "
      f"({by_age.loc[35:45].min():,}); second peak 45-60: age {by_age.loc[45:60].idxmax()} ({by_age.loc[45:60].max():,})")

customers: 1,371,980 | age missing: 15,861 (1.16%)
count    1356119.0
mean          36.4
std           14.3
min           16.0
25%           24.0
50%           32.0
75%           49.0
max           99.0
main peak: age 21 (67,530); trough 35-45 at age 39 (14,910); second peak 45-60: age 51 (29,720)


The age distribution is bimodal: a large peak in the early twenties, a dip in the late thirties and a second, lower peak around 50 (exact ages printed above).

### club_member_status and fashion_news_frequency

In [3]:
for col in ("club_member_status", "fashion_news_frequency"):
    vc = q(f"SELECT coalesce({col}, '(missing)') AS value, count(*) AS n FROM cust GROUP BY 1 ORDER BY n DESC")
    vc["share"] = (vc["n"] / vc["n"].sum()).map("{:.2%}".format)
    print(col); print(vc.to_string(index=False)); print()

club_member_status
     value       n  share
    ACTIVE 1272491 92.75%
PRE-CREATE   92960  6.78%
 (missing)    6062  0.44%
 LEFT CLUB     467  0.03%



fashion_news_frequency
    value      n  share
     NONE 877711 63.97%
Regularly 477416 34.80%
(missing)  16009  1.17%
  Monthly    842  0.06%
     None      2  0.00%



`fashion_news_frequency` has both `NONE` and `None` spellings, plus missing values; treat all three as 'no newsletter'.

### Repeat purchase and basket size
Repeat purchase = bought on at least 2 different days in the 2 years. Basket = all rows by one customer on one day (the data has no order id).

In [4]:
rep = q("""SELECT count(*) AS buyers, avg(CAST(n_days >= 2 AS DOUBLE)) AS repeat_rate,
              median(n_days) AS median_days, avg(n_days) AS mean_days, median(n_tx) AS median_rows FROM c_tx""")
never = q("SELECT avg(CAST(cust NOT IN (SELECT cust FROM c_tx) AS DOUBLE)) AS share FROM cust")["share"][0]
print(rep.round(3).to_string(index=False))
print(f"customers in customers.csv with no purchase at all: {never:.2%}")
basket = q(f"SELECT n, count(*) AS baskets FROM (SELECT cust, t_dat, count(*) AS n FROM {TX} GROUP BY 1, 2) GROUP BY n ORDER BY n")
w = basket["baskets"]
mean = (basket["n"] * w).sum() / w.sum()
cum = w.cumsum() / w.sum()
print(f"customer-days: {w.sum():,} | mean basket {mean:.2f} rows | median {basket['n'][cum >= 0.5].iloc[0]} | "
      f"p90 {basket['n'][cum >= 0.9].iloc[0]} | single-item baskets {w[basket['n'] == 1].sum() / w.sum():.1%}")

 buyers  repeat_rate  median_days  mean_days  median_rows
1362281        0.671          3.0      6.665          9.0
customers in customers.csv with no purchase at all: 0.71%


customer-days: 9,080,179 | mean basket 3.50 rows | median 2 | p90 7 | single-item baskets 29.6%


### Online vs store share and top categories by age band
Channel 2 = online, 1 = store (convention used in data.py).

In [5]:
BAND = """CASE WHEN c.age IS NULL THEN 'missing' WHEN c.age < 25 THEN '16-24' WHEN c.age < 35 THEN '25-34'
          WHEN c.age < 45 THEN '35-44' WHEN c.age < 55 THEN '45-54' ELSE '55+' END"""
con.execute(f"CREATE TEMP TABLE band AS SELECT cust, {BAND} AS age_band FROM cust c")
ch = q(f"""SELECT b.age_band, count(*) AS units, avg(CAST(t.channel = 2 AS DOUBLE)) AS online_share,
           count(DISTINCT t.cust) AS buyers
           FROM {TX} t JOIN band b USING (cust) GROUP BY 1 ORDER BY 1""")
ch["units_share"] = ch["units"] / ch["units"].sum()
print(ch.assign(online_share=ch.online_share.map("{:.1%}".format), units_share=ch.units_share.map("{:.1%}".format)).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(ch["age_band"], ch["online_share"], color=viz.SERIES[0])
ax.set_ylim(0, 1); ax.set_ylabel("online share of units"); ax.set_title("Online share by age band"); plt.show()

age_band    units online_share  buyers units_share
   16-24  6525051        66.8%  353827       20.5%
   25-34 11565068        74.8%  391009       36.4%
   35-44  4111309        72.2%  167950       12.9%
   45-54  6079556        66.7%  252409       19.1%
     55+  3367082        66.8%  181325       10.6%
 missing   140258        62.7%   15761        0.4%


In [6]:
cat = q(f"""SELECT b.age_band, a.product_type_name, count(*) AS units
            FROM {TX} t JOIN band b USING (cust) JOIN {ART} a USING (article_id) GROUP BY 1, 2""")
cat["share"] = cat["units"] / cat.groupby("age_band")["units"].transform("sum")
top = (cat.sort_values(["age_band", "share"], ascending=[True, False]).groupby("age_band").head(5)
       .assign(item=lambda d: d.product_type_name + " " + d.share.map("{:.1%}".format)))
for band, s in top.groupby("age_band")["item"]:
    print(f"{band:8} " + " | ".join(s))
idx = q(f"""SELECT b.age_band, a.index_group_name, count(*) AS units
            FROM {TX} t JOIN band b USING (cust) JOIN {ART} a USING (article_id) GROUP BY 1, 2""")
print(); print(idx.pivot(index="age_band", columns="index_group_name", values="units")
              .pipe(lambda p: p.div(p.sum(axis=1), axis=0)).map("{:.1%}".format).to_string())

16-24    Trousers 11.8% | Dress 8.7% | Sweater 8.6% | T-shirt 7.0% | Bra 5.2%
25-34    Trousers 12.3% | Dress 10.5% | Sweater 8.2% | T-shirt 6.8% | Top 4.9%
35-44    Trousers 14.0% | Dress 10.7% | Sweater 8.5% | T-shirt 7.3% | Top 4.9%
45-54    Trousers 14.7% | Dress 10.3% | Sweater 9.3% | T-shirt 7.0% | Blouse 5.2%
55+      Trousers 15.9% | Dress 11.4% | Sweater 10.5% | T-shirt 6.6% | Blouse 6.5%
missing  Trousers 13.3% | Sweater 9.4% | Dress 8.4% | T-shirt 7.9% | Top 5.0%



index_group_name Baby/Children Divided Ladieswear Menswear Sport
age_band                                                        
16-24                     1.3%   29.1%      61.1%     4.9%  3.5%
25-34                     2.5%   21.8%      66.3%     4.9%  4.5%
35-44                    10.2%   19.7%      61.2%     5.0%  3.9%
45-54                     5.4%   21.1%      62.4%     7.2%  3.8%
55+                       2.4%   17.6%      70.0%     6.9%  3.1%
missing                   7.5%   20.9%      61.5%     6.0%  4.1%


## 2. Data quality
### Exact duplicate transaction rows

In [7]:
dup = q(f"""SELECT n, count(*) AS groups FROM (SELECT t_dat, cust, article_id, price, channel, count(*) AS n
            FROM {TX} GROUP BY ALL) WHERE n > 1 GROUP BY n ORDER BY n""")
total = q(f"SELECT count(*) AS n FROM {TX}")["n"][0]
rows_in_dups = int((dup["n"] * dup["groups"]).sum())
extra = int(((dup["n"] - 1) * dup["groups"]).sum())
print(f"rows: {total:,} | rows that belong to a duplicate group: {rows_in_dups:,} ({rows_in_dups / total:.2%}) | "
      f"rows beyond the first in each group: {extra:,} ({extra / total:.2%})")
print(dup.head(8).to_string(index=False))
daily = q(f"""WITH g AS (SELECT t_dat, cust, article_id, price, channel, count(*) AS n FROM {TX} GROUP BY ALL)
             SELECT t_dat, sum(CASE WHEN n > 1 THEN n - 1 ELSE 0 END) / sum(n) AS extra_share FROM g GROUP BY 1""")["extra_share"]
print(f"share of extra duplicate rows per day: min {daily.min():.2%}, median {daily.median():.2%}, max {daily.max():.2%} "
      f"(10th-90th pct {daily.quantile(.1):.2%}-{daily.quantile(.9):.2%})")
raw = con.execute(f"""SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM
          read_csv_auto('{(RAW / 'transactions_train.csv').as_posix()}', types={{'customer_id': 'VARCHAR', 'article_id': 'VARCHAR'}})))
          FROM read_csv_auto('{(RAW / 'transactions_train.csv').as_posix()}', types={{'customer_id': 'VARCHAR', 'article_id': 'VARCHAR'}})""").fetchone()[0]
print(f"same check on the raw CSV (real customer_id, unhashed): {raw:,} extra rows (Parquet: {extra:,})")

rows: 31,788,324 | rows that belong to a duplicate group: 5,518,813 (17.36%) | rows beyond the first in each group: 2,974,905 (9.36%)
 n  groups
 2 2285073
 3  177237
 4   55412
 5    9259
 6    8181
 7    1733
 8    2132
 9     911


share of extra duplicate rows per day: min 6.11%, median 9.34%, max 15.71% (10th-90th pct 7.74%-11.10%)


same check on the raw CSV (real customer_id, unhashed): 2,974,905 extra rows (Parquet: 2,974,905)


In [8]:
mix = q(f"""WITH g AS (SELECT t_dat, cust, article_id, price, channel, count(*) AS n FROM {TX} GROUP BY ALL)
            SELECT a.product_group_name, sum(CASE WHEN n > 1 THEN n ELSE 0 END) AS dup_rows, sum(n) AS all_rows
            FROM g JOIN {ART} a USING (article_id) GROUP BY 1 ORDER BY dup_rows DESC LIMIT 8""")
mix["share_of_group_rows_duplicated"] = (mix["dup_rows"] / mix["all_rows"]).map("{:.1%}".format)
print(mix.to_string(index=False))
px = q(f"""WITH g AS (SELECT t_dat, cust, article_id, price, channel, count(*) AS n FROM {TX} GROUP BY ALL)
          SELECT CASE WHEN n > 1 THEN 'duplicated' ELSE 'single' END AS kind, median(price) AS median_price, count(*) AS groups
          FROM g GROUP BY 1""")
print(); print(px.to_string(index=False))

product_group_name  dup_rows   all_rows share_of_group_rows_duplicated
Garment Upper body 1843860.0 12552755.0                          14.7%
Garment Lower body 1472014.0  7046054.0                          20.9%
          Swimwear  710493.0  2579222.0                          27.5%
 Garment Full body  664104.0  3552470.0                          18.7%
         Underwear  414885.0  2565858.0                          16.2%
    Socks & Tights  128147.0   685712.0                          18.7%
       Accessories  109992.0  1599593.0                           6.9%
             Shoes  105553.0   745521.0                          14.2%



      kind  median_price   groups
duplicated      0.025407  2543908
    single      0.025407 26269511


**Reading:** the dataset has no quantity column, and every duplicate row has the same customer, day, article,
price *and* channel. The count of groups falls steeply with group size (2 ≫ 3 ≫ 4), with small bumps at even counts
(6, 8) that fit buying in pairs. The share of duplicate rows is spread over every day (printed range above) rather
than concentrated on a few days, which argues against a one-off logging fault, and the median price of duplicated
and single rows is the same. The raw CSV with the real customer_id gives the same count as the Parquet file, so it
is not caused by hashing. We cannot prove it without an order id or quantity field, but multiple units bought in one
transaction is the most consistent explanation. We treat each row as one unit sold: weekly `units = count(*)` keeps
them, and dropping them would undercount multi-buys.

### Prices: scaling and outliers

In [9]:
p = q(f"SELECT price FROM {TX} USING SAMPLE 2000000 ROWS (reservoir, 42)")["price"]
print(p.describe(percentiles=[.01, .5, .99]).to_string())
po = q(f"""WITH x AS (SELECT a.product_type_name, ln(t.price) AS lp FROM {TX} t JOIN {ART} a USING (article_id) WHERE t.price > 0),
           q AS (SELECT product_type_name, quantile_cont(lp, 0.25) AS q1, quantile_cont(lp, 0.75) AS q3 FROM x GROUP BY 1)
           SELECT count(*) AS rows, sum(CAST(lp > q3 + 3 * (q3 - q1) AS INT)) AS high, sum(CAST(lp < q1 - 3 * (q3 - q1) AS INT)) AS low
           FROM x JOIN q USING (product_type_name)""")
print(po.assign(high_share=po.high / po.rows, low_share=po.low / po.rows).to_string(index=False))
print("rows with price <= 0:", q(f"SELECT count(*) AS n FROM {TX} WHERE price <= 0")["n"][0])

count    2.000000e+06
mean     2.781485e-02
std      1.916344e-02
min      3.389830e-05
1%       3.593220e-03
50%      2.540678e-02
99%      1.001356e-01
max      5.915254e-01


    rows    high     low  high_share  low_share
31788324 12950.0 81913.0    0.000407   0.002577
rows with price <= 0: 0


Prices are **scaled** by Kaggle (summary above: max ≈ 0.59, median ≈ 0.025), not in a currency. Ratios within the data are
meaningful (the model's `discount` = 1 − price / highest price seen), absolute levels are not. Outliers are flagged
on log price within each product type (beyond 3 × IQR). There are no zero or negative prices. Low outliers (0.26% of rows;
high outliers 0.04%) are kept: they are consistent with markdowns, and without a price list there is no way to
tell them from errors.

### Missing values: article descriptions and customer age

In [10]:
print(q(f"SELECT count(*) AS articles, sum(CAST(detail_desc IS NULL OR trim(detail_desc) = '' AS INT)) AS missing_detail_desc FROM {ART}")
      .assign(share=lambda d: (d.missing_detail_desc / d.articles).map("{:.2%}".format)).to_string(index=False))
print(q("SELECT count(*) AS customers, sum(CAST(age IS NULL AS INT)) AS missing_age FROM cust")
      .assign(share=lambda d: (d.missing_age / d.customers).map("{:.2%}".format)).to_string(index=False))

 articles  missing_detail_desc share
   105542                416.0 0.39%
 customers  missing_age share
   1371980      15861.0 1.16%


### Articles and styles with no sales

In [11]:
ns = q(f"""WITH s AS (SELECT DISTINCT article_id FROM {TX})
          SELECT count(*) AS articles, sum(CAST(s.article_id IS NULL AS INT)) AS articles_no_sales,
                 count(DISTINCT a.product_code) AS styles,
                 count(DISTINCT a.product_code) FILTER (WHERE a.product_code NOT IN
                     (SELECT lpad(CAST(article_id // 1000 AS VARCHAR), 7, '0') FROM s)) AS styles_no_sales
          FROM {ART} a LEFT JOIN s USING (article_id)""")
print(ns.to_string(index=False))

 articles  articles_no_sales  styles  styles_no_sales
   105542              995.0   47224              390


### Is product_code (our style id) stable across its articles?

In [12]:
print(q(f"SELECT count(*) AS articles, sum(CAST(article_id // 1000 <> CAST(product_code AS INT) AS INT)) AS id_mismatch FROM {ART}").to_string(index=False))
cols = ["prod_name", "product_type_name", "product_group_name", "garment_group_name", "index_group_name", "department_name", "colour_group_name"]
st = q(f"SELECT {', '.join(f'count(DISTINCT {c}) AS {c}' for c in cols)}, count(*) AS n_articles FROM {ART} GROUP BY product_code")
print(f"styles: {len(st):,} | articles per style: mean {st.n_articles.mean():.2f}, max {st.n_articles.max()}")
print("share of styles with more than one value:")
print((st[cols] > 1).mean().map("{:.2%}".format).to_string())

 articles  id_mismatch
   105542          0.0
styles: 47,224 | articles per style: mean 2.23, max 75
share of styles with more than one value:
prod_name              4.61%
product_type_name      0.66%
product_group_name     0.20%
garment_group_name     0.29%
index_group_name       0.19%
department_name        0.98%
colour_group_name     47.33%


`article_id // 1000 = product_code` for every article, so the style id is consistent. Colour varies within a
style by design (articles = colour variants). Product type and garment group are stable for almost all styles; where
a style has more than one, the model uses the most common value (`data._attribute_modes`).

### Date coverage

In [13]:
d = q(f"SELECT t_dat, count(*) AS rows FROM {TX} GROUP BY 1 ORDER BY 1")
d["t_dat"] = pd.to_datetime(d["t_dat"])
full = pd.date_range(d.t_dat.min(), d.t_dat.max())
missing = full.difference(d.t_dat)
print(f"{d.t_dat.min():%Y-%m-%d} → {d.t_dat.max():%Y-%m-%d}: {len(full)} days, {len(d)} with data, missing: {list(missing.strftime('%Y-%m-%d'))}")
print("lowest-volume days:"); print(d.nsmallest(5, "rows").assign(t_dat=lambda x: x.t_dat.dt.strftime('%Y-%m-%d')).to_string(index=False))
print("highest-volume days:"); print(d.nlargest(5, "rows").assign(t_dat=lambda x: x.t_dat.dt.strftime('%Y-%m-%d')).to_string(index=False))

2018-09-20 → 2020-09-22: 734 days, 734 with data, missing: []
lowest-volume days:
     t_dat  rows
2020-01-01 12760
2019-12-25 13637
2018-12-25 14799
2018-09-22 17635
2019-01-01 18145
highest-volume days:
     t_dat   rows
2019-09-28 198622
2020-04-11 162799
2019-11-29 160875
2018-11-23 142018
2018-09-29 141700


## 3. Sampling note and the bias it introduces

- **Used in full:** all 31.8M transactions, all 105,542 articles and all 1.37M customers. No row sampling, except a
  2M-row reservoir sample (seed 42) for the price summary statistics; the price outlier counts use every row.
- **Images:** only the 9 reference photos of the 3 winning styles (3 colours each) were downloaded. The full image
  set is about 30 GB and the forecast does not use images.
- **Bias:** the forecast is image-free, so nothing in the ranking is biased by the photos. The concepts are, though:
  they reflect only the best-selling colours of 3 styles, so colours and styles that were not chosen never reach the
  design step. The image critic was also only checked against these 9 photos.
- **Data biases to keep in mind:** sales = demand only when items were in stock (no stock data); `customers.csv`
  age is self-reported and 1.2% missing; one customer-day is treated as one basket because there is no order id.